# Simple RAG (Retrieval-Augmented Generation)
**Knowledge Base:** `llm_kb.txt` — a short text file about LLMs

**Stack:**
- `sentence-transformers` → embed text into vectors
- `faiss` → store & search vectors
- `transformers` (flan-t5-base) → generate an answer from retrieved context

In [5]:
import torch

if torch.cuda.is_available():
    device = 'cuda'
    print(f'Using GPU: {torch.cuda.get_device_name(0)}')
else:
    device = 'cpu'
    print('Using CPU')

Using CPU


## Step 1 — Load and chunk the knowledge base

In [6]:
with open('llm_kb.txt', 'r', encoding='utf-8') as f:
    raw = f.read()

# Split into paragraphs (each paragraph = one chunk)
chunks = [p.strip() for p in raw.split('\n\n') if p.strip()]

print(f'Total chunks: {len(chunks)}')
print('--- First chunk ---')
print(chunks[0])

Total chunks: 15
--- First chunk ---
Large Language Models (LLMs) are a type of artificial intelligence trained on massive amounts of text data. They learn to predict the next word in a sequence, which forces them to develop a deep understanding of language, facts, and reasoning. Examples include GPT-4, Claude, and Gemini.


## Step 2 — Embed all chunks with sentence-transformers

In [7]:
from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

embeddings = embed_model.encode(chunks, convert_to_numpy=True, show_progress_bar=True)
print(f'Embeddings shape: {embeddings.shape}')  # (num_chunks, 384)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (15, 384)


## Step 3 — Build a FAISS index

In [8]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)          # simple L2 distance index
index.add(embeddings.astype('float32'))

print(f'Vectors in index: {index.ntotal}')

Vectors in index: 15


## Step 4 — Retrieve relevant chunks for a query

In [9]:
def retrieve(query, top_k=3):
    query_vec = embed_model.encode([query], convert_to_numpy=True).astype('float32')
    distances, indices = index.search(query_vec, top_k)
    results = []
    for rank, (idx, dist) in enumerate(zip(indices[0], distances[0])):
        results.append({'rank': rank + 1, 'score': float(dist), 'chunk': chunks[idx]})
    return results

# Try it
query = 'What is hallucination in LLMs?'
hits = retrieve(query)

print(f'Query: {query}\n')
for h in hits:
    print(f"[Rank {h['rank']} | L2={h['score']:.2f}]")
    print(h['chunk'])
    print()

Query: What is hallucination in LLMs?

[Rank 1 | L2=0.49]
Hallucination is when an LLM confidently generates text that is factually wrong. This happens because the model is trained to predict plausible-sounding tokens, not to retrieve verified facts. For example, an LLM might confidently cite a research paper that doesn't exist.

[Rank 2 | L2=1.23]
Large Language Models (LLMs) are a type of artificial intelligence trained on massive amounts of text data. They learn to predict the next word in a sequence, which forces them to develop a deep understanding of language, facts, and reasoning. Examples include GPT-4, Claude, and Gemini.

[Rank 3 | L2=1.24]
Retrieval-Augmented Generation (RAG) is a technique that addresses hallucination by giving the LLM access to a relevant external knowledge base at inference time. Instead of relying only on what's baked into the model's weights, RAG retrieves relevant passages and includes them in the prompt. This grounds the model's response in real docum

## Step 5 — Generate an answer using the retrieved context (flan-t5-base)

We stuff the top-k chunks into the prompt and let a small local model answer.

In [10]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base').to(device)
print('Generator ready.')

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Generator ready.


In [11]:
def rag(query, top_k=3, max_new_tokens=200):
    hits = retrieve(query, top_k=top_k)
    context = '\n\n'.join(h['chunk'] for h in hits)

    prompt = (
        f'Answer the question based only on the context below.\n\n'
        f'Context:\n{context}\n\n'
        f'Question: {query}\n'
        f'Answer:'
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=512).to(device)
    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

answer = rag('What is hallucination in LLMs?')
print(answer)

when an LLM confidently generates text that is factually wrong


In [12]:
# Try more questions
for q in [
    'What is RAG and why is it useful?',
    'How does self-attention work?',
    'What is quantization?',
]:
    print(f'Q: {q}')
    print(f'A: {rag(q)}')
    print()

Q: What is RAG and why is it useful?
A: RAG is a technique that addresses hallucination by giving the LLM access to a relevant external knowledge base at inference time

Q: How does self-attention work?
A: allows a model to weigh how important each token is relative to every other token in the input

Q: What is quantization?
A: a technique to reduce the memory footprint of LLMs

